In [16]:
import pandas as pd
import math
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

In [17]:
df = pd.read_csv('data NTPT.csv')
df.head()

,bulan tahun,NTPT
0,Juli 2017,104.08
1,Agustus 2017,104.56
2,September 2017,104.45
3,Oktober 2017,103.20
4,November 2017,104.72


In [18]:
# min
# max
# z1, z2
# N
# R = Dmax + Z2 - (Dmin - Z1)
# mean = sum |Dt+1 - Dt|/N-1
# K = mean/2
# n = R/K

In [19]:
Dt = df['NTPT']
Dmin = Dt.min()
Dmax = Dt.max()
N = Dt.count()
Z1 = 0.2
Z2 = 0.7

print(Dmin, Dmax, N)

103.2 113.59 30


In [20]:
R = Dmax+Z2 - (Dmin-Z1)
print(R)

11.290000000000006


In [21]:
dif = []

for i in range (N-1):
    dt = df['NTPT'][i]
    dt2 = df['NTPT'][i+1]
    ab = round(abs(dt2-dt),2)
    dif.append(ab)
dif.append(0)

df2 = df
df2['|Dt+1 - Dt|'] = dif

df2.head()

,bulan tahun,NTPT,|Dt+1 - Dt|
0,Juli 2017,104.08,0.48
1,Agustus 2017,104.56,0.11
2,September 2017,104.45,1.25
3,Oktober 2017,103.20,1.52
4,November 2017,104.72,1.58


In [22]:
mean = sum(df2['|Dt+1 - Dt|'])/(N-1)
print(mean)

0.9324137931034484


In [23]:
K = mean/2
if(K<1):
    K = round(K,1)
else:
    K = math.ceil(K)
print(K)

0.5


In [24]:
# math.ceil = roundup
n = math.ceil(R/K)
print(n)


23


In [25]:
# himpunan fuzzy
# batas atas, batas bawah, nilai tengah

In [26]:
himpFuzzy = pd.DataFrame()
ui = []
batasBawah = []
batasAtas = []
Ai = []
ui.append('u1')
Ai.append('A1')
batasBawah.append(Dmin-Z1)
batasAtas.append(batasBawah[0] + K)

for i in range(1, n):
    ui.append('u'+str(i+1))
    Ai.append('A'+str(i+1))
    batasBawah.append(batasAtas[i-1])
    batasAtas.append(batasBawah[i] + K)

himpFuzzy['ui'] = ui
himpFuzzy['Ai'] = Ai
himpFuzzy['batas bawah'] = batasBawah
himpFuzzy['batas atas'] = batasAtas

In [27]:
himpFuzzy.head()

,ui,Ai,batas bawah,batas atas
0,u1,A1,103.0,103.5
1,u2,A2,103.5,104.0
2,u3,A3,104.0,104.5
3,u4,A4,104.5,105.0
4,u5,A5,105.0,105.5


In [28]:
# mi / nilai tengah
himpFuzzy['mi'] = (himpFuzzy['batas bawah'] + himpFuzzy['batas atas'])/2
himpFuzzy.head()

,ui,Ai,batas bawah,batas atas,mi
0,u1,A1,103.0,103.5,103.25
1,u2,A2,103.5,104.0,103.75
2,u3,A3,104.0,104.5,104.25
3,u4,A4,104.5,105.0,104.75
4,u5,A5,105.0,105.5,105.25


In [29]:
himpFuzzy.iloc[[0,1,2,len(himpFuzzy.index)-1]]

,ui,Ai,batas bawah,batas atas,mi
0,u1,A1,103.0,103.5,103.25
1,u2,A2,103.5,104.0,103.75
2,u3,A3,104.0,104.5,104.25
22,u23,A23,114.0,114.5,114.25


In [30]:
data = df2['NTPT']

In [31]:
# FLR dan FLRG
hasilFuzzy = []
for i in data:
    copyHimp = himpFuzzy
    cond1 = copyHimp['batas bawah'] <= i 
    cond2 = copyHimp['batas atas'] >= i 
    dfAi = copyHimp.where(cond1 & cond2) 
    dfAi = dfAi[~dfAi['ui'].isna()]
    fuzzyfikasi = dfAi['Ai'].iloc[0]
    hasilFuzzy.append(fuzzyfikasi)

df2['fuzzyfikasi'] = hasilFuzzy

In [32]:
df2.head()

,bulan tahun,NTPT,|Dt+1 - Dt|,fuzzyfikasi
0,Juli 2017,104.08,0.48,A3
1,Agustus 2017,104.56,0.11,A4
2,September 2017,104.45,1.25,A3
3,Oktober 2017,103.20,1.52,A1
4,November 2017,104.72,1.58,A4


In [33]:
nextState = []
for k in range(0, len(df2.index)-1):
    neSt = df2['fuzzyfikasi'].iloc[k+1]
    nextState.append(neSt)

# nextState.append('')

df2['next state'] = pd.Series(nextState)
df2.head()

,bulan tahun,NTPT,|Dt+1 - Dt|,fuzzyfikasi,next state
0,Juli 2017,104.08,0.48,A3,A4
1,Agustus 2017,104.56,0.11,A4,A3
2,September 2017,104.45,1.25,A3,A1
3,Oktober 2017,103.20,1.52,A1,A4
4,November 2017,104.72,1.58,A4,A7


In [34]:
df2.iloc[[0,1,2,len(df2.index)-1]]

,bulan tahun,NTPT,|Dt+1 - Dt|,fuzzyfikasi,next state
0,Juli 2017,104.08,0.48,A3,A4
1,Agustus 2017,104.56,0.11,A4,A3
2,September 2017,104.45,1.25,A3,A1
29,Desember 2020,111.18,0.00,A17,NaN


In [35]:
# FLRG
dfFLRG = pd.DataFrame()
dfFLRG['Ai'] = himpFuzzy['Ai']
dfFLRG['FLRG'] = ''

# ambil df2 kecuali baris terakhir karena next statenya kosong
df3 = df2[:-1]

for f in range(len(dfFLRG.index)):
    searchAi = 'A'+str(f+1)
    
    new = df3[df3['fuzzyfikasi'].isin([searchAi])]
    group = new['next state']
    group = group.to_numpy()
    
    if len(group) > 0 :
        dfFLRG['FLRG'][f] = group

dfFLRG

,Ai,FLRG
0,A1,[A4]
1,A2,
2,A3,"[A4, A1]"
3,A4,"[A3, A7]"
4,A5,
5,A6,
6,A7,"[A11, A9]"
7,A8,
8,A9,"[A7, A12]"
9,A10,


In [36]:
dfFLRG.iloc[[0,1,2,len(dfFLRG.index)-1]]

,Ai,FLRG
0,A1,[A4]
1,A2,
2,A3,"[A4, A1]"
22,A23,


In [37]:
# hasil fuzzyfikasi
dfFLRG['defuzzyfikasi'] = ''

for g in range(len(dfFLRG.index)):
    ai = dfFLRG['Ai'].loc[g]
    flrg = dfFLRG['FLRG'].loc[g]
    hasil = 0

    if (len(flrg) == 0):
        fuzzyfikasi = himpFuzzy[himpFuzzy['Ai'] == ai]['mi']
        hasil = fuzzyfikasi.values[0]
    
    elif (len(flrg) == 1):
        fuzzyfikasi = himpFuzzy[himpFuzzy['Ai'] == flrg[0]]['mi']
        hasil = fuzzyfikasi.values[0]
        
    else:
        lenFlrg = len(flrg)
        for h in range(len(flrg)):
            mi = himpFuzzy[himpFuzzy['Ai'] == flrg[h]]['mi']
            bobot = (1/len(flrg)) * mi.values[0]
            hasil = hasil + bobot
            

    dfFLRG['defuzzyfikasi'][g] = hasil

dfFLRG


,Ai,FLRG,defuzzyfikasi
0,A1,[A4],104.75
1,A2,,103.75
2,A3,"[A4, A1]",104.0
3,A4,"[A3, A7]",105.25
4,A5,,105.25
5,A6,,105.75
6,A7,"[A11, A9]",107.75
7,A8,,106.75
8,A9,"[A7, A12]",107.5
9,A10,,107.75


In [38]:
dfFLRG.iloc[[0,1,2,len(dfFLRG.index)-1]]

,Ai,FLRG,defuzzyfikasi
0,A1,[A4],104.75
1,A2,,103.75
2,A3,"[A4, A1]",104.0
22,A23,,114.25


In [39]:
df2['yt'] = ''

for y in range (len(dfFLRG.index)):
    ai = dfFLRG['Ai'].loc[y]
    listAi = df2[df2['fuzzyfikasi'] == ai]
    yt = dfFLRG['defuzzyfikasi'][y]
    
    if len(listAi) > 0:
        idx = listAi.index
        for z in idx:
            df2.at[z+1,'yt'] = yt

In [40]:
df2

,bulan tahun,NTPT,|Dt+1 - Dt|,fuzzyfikasi,next state,yt
0,Juli 2017,104.08,0.48,A3,A4,
1,Agustus 2017,104.56,0.11,A4,A3,104.0
2,September 2017,104.45,1.25,A3,A1,105.25
3,Oktober 2017,103.20,1.52,A1,A4,104.0
4,November 2017,104.72,1.58,A4,A7,104.75
5,Desember 2017,106.30,1.90,A7,A11,105.25
6,Januari 2018,108.20,0.96,A11,A9,107.75
7,Februari 2018,107.24,0.85,A9,A7,107.25
8,Maret 2018,106.39,0.77,A7,A9,107.5
9,April 2018,107.16,1.66,A9,A12,107.75


In [45]:
df2.iloc[[0,1,2,len(df2.index)-2,len(df2.index)-1]]

,bulan tahun,NTPT,|Dt+1 - Dt|,fuzzyfikasi,next state,yt
0,Juli 2017,104.08,0.48,A3,A4,
1,Agustus 2017,104.56,0.11,A4,A3,104.0
2,September 2017,104.45,1.25,A3,A1,105.25
29,Desember 2020,111.18,0.00,A17,NaN,110.625
30,NaN,NaN,NaN,NaN,NaN,110.25


In [48]:
# MAPE
y_actual = df2['NTPT'][1:-1].to_numpy()
# print(y_actual)
y_predict = df2['yt'][1:-1].to_numpy()
# print(y_predict)
Mape = mean_absolute_percentage_error(y_actual, y_predict)
MSE = mean_squared_error(y_actual, y_predict, squared=False)
MAE = mean_absolute_error(y_actual, y_predict)
print("MAPE =",str(Mape*100)+"%")
print("MSE =",MSE)
print("MAPE =",MAE)

MAPE = 0.5341973680689682%
MSE = 0.7694291275133851
MAPE = 0.5826600985221668


In [43]:
# # prediksi 5 periode berikutnya = 4 karena yg 1 udh diatas

# nextYt = 10
# lastIndex = len(df2.index)
# dfPredictNext = df2

# # print(len(df2.index))
# for n in range (nextYt-1):
#     # Add fuzzyfikai dr hasil predict
#     nilaiYt = dfPredictNext['yt'][lastIndex-1+n]
    
#     copyHimp = himpFuzzy
#     cond1 = copyHimp['batas bawah'] <= nilaiYt
#     cond2 = copyHimp['batas atas'] >= nilaiYt
#     dfAi = copyHimp.where(cond1 & cond2) 
#     dfAi = dfAi[~dfAi['ui'].isna()]
#     fuzzyfikasi = dfAi['Ai'].iloc[0]
#     dfPredictNext.at[lastIndex + n -1, 'fuzzyfikasi'] = fuzzyfikasi
#     # print(fuzzyfikasi)

#     # add predict
#     yt = dfFLRG['defuzzyfikasi'][dfFLRG['Ai'] == fuzzyfikasi]
#     dfPredictNext.at[lastIndex+n, 'yt'] = yt.values[0]
#     # print(yt.values[0])
    

In [44]:
# df2.drop(index=df2.index[-1],axis=0,inplace=True)
# dfPredictNext